# K-Visitation worked example

This notebook reproduces the hypothetical 12-cell example in the current Supplementary Information. It constructs the recurrent set $K_{freq}$, the nearest set $K_{dist}$ and their visit-frequency-weighted overlap $q_K$. The sample contains no observed user or location data.

In [1]:
from pathlib import Path
import sys
import pandas as pd

candidates = [Path.cwd(), Path.cwd() / 'figure-reproduction', Path.cwd().parent]
repro_dir = next(
    (path.resolve() for path in candidates if (path / 'src').is_dir() and (path / 'data').is_dir()),
    None,
)
if repro_dir is None:
    raise FileNotFoundError('Run this notebook from the repository, figure-reproduction, or notebooks directory.')
sys.path.insert(0, str(repro_dir / 'src'))

from k_visitation import DEFAULT_AMENITY_COLUMNS, calculate_k_visitation

In [2]:
portfolio = pd.read_csv(repro_dir / 'data' / 'sample_data_k_visitation.csv')
portfolio[['cell_id', 'visit_freq', 'home_dist_km', *DEFAULT_AMENITY_COLUMNS]]

,cell_id,visit_freq,home_dist_km,CIVIC_RELIGION,CULTURE,DINING,EDUCATION,FITNESS,GROCERIES,HEALTHCARE,RETAIL,SERVICE,TRANSPORT
0,a,60,4.5,0,1,1,0,1,0,0,1,1,0
1,b,48,2.1,0,0,0,0,0,0,1,1,0,0
2,c,40,0.6,1,0,0,1,0,0,0,0,0,0
3,d,30,0.7,0,0,1,0,0,1,0,0,1,0
4,e,22,5.2,0,1,1,0,0,0,0,0,0,1
5,f,15,0.1,0,0,0,0,0,0,0,0,1,1
6,g,12,6.2,1,1,1,1,0,0,0,1,1,1
7,h,9,6.9,0,0,0,0,0,0,0,0,1,0
8,i,8,1.8,0,0,0,0,0,1,0,0,0,1
9,j,5,0.6,0,1,0,0,0,0,0,0,0,0


In [3]:
places, user_qk = calculate_k_visitation(
    portfolio,
    location_col='cell_id',
    distance_col='home_dist_km',
)
places[['cell_id', 'visit_freq', 'home_dist_km', 'k_freq', 'k_dist', 'k_type']]

,cell_id,visit_freq,home_dist_km,k_freq,k_dist,k_type
0,a,60,4.5,1,0,f1d0
1,b,48,2.1,1,1,f1d1
2,c,40,0.6,1,1,f1d1
3,d,30,0.7,1,1,f1d1
4,e,22,5.2,1,0,f1d0
5,f,15,0.1,0,1,f0d1
6,g,12,6.2,0,0,f0d0
7,h,9,6.9,0,0,f0d0
8,i,8,1.8,0,1,f0d1
9,j,5,0.6,0,1,f0d1


In [4]:
k_freq = places.loc[places['k_freq'].eq(1), 'cell_id'].tolist()
k_dist = (
    places.loc[places['k_dist'].eq(1)]
    .sort_values(['home_dist_km', 'cell_id'])['cell_id']
    .tolist()
)
assert k_freq == list('abcde')
assert k_dist == list('fkcjdib')
assert user_qk.loc[0, 'intersection_visits'] == 118
assert user_qk.loc[0, 'union_visits'] == 232
assert abs(user_qk.loc[0, 'qk'] - 118 / 232) < 1e-12
print(f"K_freq = {k_freq}")
print(f"K_dist = {k_dist}")
print(f"q_K = 118 / 232 = {user_qk.loc[0, 'qk']:.4f}")
user_qk

K_freq = ['a', 'b', 'c', 'd', 'e']
K_dist = ['f', 'k', 'c', 'j', 'd', 'i', 'b']
q_K = 118 / 232 = 0.5086


,user_id,k_freq_size,k_dist_size,intersection_size,union_size,intersection_visits,union_visits,qk,qk_unweighted
0,toy_user,5,7,3,9,118,232,0.508621,0.333333
